# FIONA Treatment Example

This notebook tests `shrecc.tyndp` and `shrecc.premise_mapping` from TYNDP scenario parameters. The user provides the scenario, model year, climate year, and local storage folder; SHRECC/FIONA then loads cached pickles, parses a cached TYNDP workbook, or downloads and extracts the official zip file as needed. It builds `Z_gross`, computes the hourly consumption mix, and maps it to premise regions and activities.

In [1]:
import sys
from pathlib import Path
from importlib.resources import files

import numpy as np
import xarray as xr

from shrecc.tyndp import (
    build_z_gross_from_tyndp_scenario,
    consumption_mix_from_tyndp_scenario,
    consumption_mix_from_z_gross,
    tyndp_scenario_paths,
)

from shrecc.premise_mapping import (
    PremiseConsumptionMixMapper,
    build_premise_region_map,
)

## Declare scenario parameters

In [2]:
scenario = "DE"  # DE = Distributed Energy, GA = Global Ambition, NT = National Trends
year = 2040  # DE/GA: 2035, 2040, 2050; NT: 2030, 2040
climate_year = 2009  # 1995, 2008, 2009

![image.png](tyndp_story_lines_screenshot.png)

## Configure storage and mappings

In [3]:
# Local cache directory for TYNDP workbooks, zip files, and generated pickles
data_dir = Path.cwd() / "data" / "tyndp"

# These SHRECC/FIONA mapping files are included in the package
technology_mapping = files("shrecc") /"data" / "tyndp" / "tyndp_ei_mapping.xlsx"
country_mapping = files("shrecc") / "data" / "tyndp" / "tyndp_country_mapping.xlsx"

for path in [technology_mapping, country_mapping]:
    assert path.exists(), path

paths = tyndp_scenario_paths(data_dir, scenario, year, climate_year)
paths

{'production_pickle': WindowsPath('d:/shrecc_project/shrecc/notebooks/data/tyndp/DE2040_CY2009_prod.pkl'),
 'trade_pickle': WindowsPath('d:/shrecc_project/shrecc/notebooks/data/tyndp/DE2040_CY2009_trade.pkl'),
 'workbook': WindowsPath('d:/shrecc_project/shrecc/notebooks/data/tyndp/MMStandardOutputFile_DE2040_Plexos_CY2009_v11_SoS.xlsb'),
 'zip': WindowsPath('d:/shrecc_project/shrecc/notebooks/data/tyndp/DE2040CY2009.zip'),
 'url': 'https://2024-data.entsos-tyndp-scenarios.eu/files/scenarios-outputs/DE2040CY2009.zip'}

## Build `Z_gross`

`Z_gross` is the gross hourly supply table with production and trade flows in one column structure. The scenario helper uses the fastest available input: existing production/trade pickles first, then an existing `.xlsb` workbook, then the official TYNDP scenario zip download.

In [4]:
Z_gross = build_z_gross_from_tyndp_scenario(
    scenario=scenario,
    year=year,
    climate_year=climate_year,
    data_dir=data_dir,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    download=True,
    verbose=True,
    engine="calamine",
)

Z_gross.shape, Z_gross.columns.names, Z_gross.index[:3]

2026-05-18 16:00:06 Loading TYNDP pickle files
2026-05-18 16:00:07 Parsing hourly datetime index
2026-05-18 16:00:07 Loading technology and country mappings
2026-05-18 16:00:08 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\tyndp.py:1107: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-18 16:00:08 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\tyndp.py:297: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\tyndp.py:1169: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-18 16:00:08 Combining production and trade into Z_gross
2026-05-18 16:00:08 Built Z_gross with shape (8736, 512)


((8736, 512),
 FrozenList(['type', 'country from', 'country to', 'source']),
 DatetimeIndex(['2040-01-01 00:00:00', '2040-01-01 01:00:00',
                '2040-01-01 02:00:00'],
               dtype='datetime64[ns]', freq=None))

In [5]:
assert len(Z_gross) > 0
assert Z_gross.columns.names == ["type", "country from", "country to", "source"]
assert {"production mix", "trade"}.issubset(Z_gross.columns.get_level_values("type"))

Z_gross.head(2)

type                   production mix                                     \
country from                       AL                                      
country to                         AL                                      
source              Gas CCGT new [MW] Gas CCGT old 2 [MW] Reservoir [MW]   
2040-01-01 00:00:00               170                   0           1904   
2040-01-01 01:00:00               160                   0           1904   

type                                                             \
country from                                                      
country to                                                        
source              Run-of-River [MW] Solar (Photovoltaic) [MW]   
2040-01-01 00:00:00               193                         0   
2040-01-01 01:00:00               193                         0   

type                                                      \
country from                                          AT   
country to                                            AT   
source              Wind Onshore [MW] Hydrogen CCGT [MW]   
2040-01-01 00:00:00                40               2934   
2040-01-01 01:00:00                43               2934   

type                                                                 \
country from                                                          
country to                                                            
source              Others non-renewable [MW] Others renewable [MW]   
2040-01-01 00:00:00                       464                   502   
2040-01-01 01:00:00                       464                   502   

type                                                           ... trade  \
country from                                                   ...    SI   
country to                                                     ...    IT   
source              Pump Storage - Closed Loop (turbine) [MW]  ... trade   
2040-01-01 00:00:00                                         0  ...     0   
2040-01-01 01:00:00                                         0  ...     0   

type                                                                       
country from           SK                UK                                
country to             CZ    HU    PL    BE    DE    DK    FR    IE    NL  
source              trade trade trade trade trade trade trade trade trade  
2040-01-01 00:00:00  2600     0     0  1000  2800     0     0     0  1000  
2040-01-01 01:00:00  2600     0     0  1000  2800     0     0     0  1000  

[2 rows x 512 columns]

## Compute Consumption Mix From `Z_gross`

The debug output includes the direct production/trade coefficient matrices used for the conservation checks.

In [6]:
consumption_mix_xr, debug = consumption_mix_from_z_gross(
    Z_gross,
    check=True,
    return_debug=True,
    # for some reason, it is possible to have no production, no imports, at certain hours
    # what to do when that happens:
    # - "raise": raise an error (default)
    # - "month_hour_average": fill with the average consumption mix for that month and hour
    zero_consumption="month_hour_average", 
    verbose=True,
)

consumption_mix_xr

2026-05-18 16:00:27 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\tyndp.py:1273: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\tyndp.py:1293: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-18 16:00:27 Calculating direct production and trade shares
2026-05-18 16:00:27 Using 35 countries and 31 technologies
2026-05-18 16:00:27 Reshaping production shares
2026-05-18 16:00:28 Reshaping trade shares
2026-05-18 16:00:28 Checking direct shares sum to one
2026-05-18 16:00:28 Solving hourly trade system
2026-05-18 16:00:39 Imputing zero-consumption country-hours with month-hour averages
2026-05-18 16:00:39 Checking final consumption mix sums to one
2026-05-18 16:00:40 Building xarray DataArray
2026-05-18 16:00:51 Built consumption_mix_xr with shape (8736, 35, 35, 31)


<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     source_country: 35, technology: 31)> Size: 3GB
array([[[[0.00000000e+00, 3.88216488e-02, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 9.13450559e-03],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 2.31019124e-04, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.43574409e-05],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.14460710e-01],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.49265824e-03],
...
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.16312976e-01],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 9.11805414e-04, 6.47051738e-04]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 4.29978847e-01, 3.05129314e-01]]]])
Coordinates:
  * source_country    (source_country) object 280B 'AL' 'AT' 'BA' ... 'SK' 'UK'
  * technology        (technology) object 248B 'Gas CCGT CCS [MW]' ... 'Wind ...
  * time              (time) datetime64[ns] 70kB 2040-01-01 ... 2040-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'

In [7]:
np.testing.assert_allclose(
    debug["direct_total"][~debug["zero_consumption_mask"]],
    1,
    atol=1e-8,
)

In [8]:
# Consumption mix in Luxembourg for the first modelled hour
first_hour = f"{year}-03-01T05:00:00"

consumption_mix_xr.sel(
    consumer_country="AL",
    time=first_hour,
).to_dataframe()["consumption_mix"].nlargest(30)

source_country  technology                               
AL              Reservoir [MW]                               0.636872
                Run-of-River [MW]                            0.086672
                Wind Onshore [MW]                            0.077560
GR              Wind Onshore [MW]                            0.075445
                Wind Offshore [MW]                           0.053918
ME              Reservoir [MW]                               0.009065
GR              Others renewable [MW]                        0.008889
                Reservoir [MW]                               0.006822
IT              Wind Onshore [MW]                            0.005989
                Wind Offshore [MW]                           0.005505
GR              Pump Storage - Open Loop (turbine) [MW]      0.003981
ME              Wind Onshore [MW]                            0.002874
GR              Run-of-River [MW]                            0.002669
IT              Solar (Photovolt

In [9]:
direct_total = debug["direct_total"]

direct_total[~debug["zero_consumption_mask"]].min(), direct_total.max()

(0.9999999999999997, 1.0000000000000004)

In [10]:
mix_total = consumption_mix_xr.sum(["source_country", "technology"])

float(mix_total.min()), float(mix_total.max())

(0.9999999999999996, 1.0000000000000004)

In [11]:
np.testing.assert_allclose(direct_total[~debug["zero_consumption_mask"]], 1, atol=1e-8)
np.testing.assert_allclose(mix_total, 1, atol=1e-8)

print("All conservation checks passed.")

All conservation checks passed.


## One-Step Pipeline

The scenario convenience wrapper should give the same result as building `Z_gross` explicitly first. It follows the same storage fallback order.

In [12]:
consumption_mix_direct = consumption_mix_from_tyndp_scenario(
    scenario=scenario,
    year=year,
    climate_year=climate_year,
    data_dir=data_dir,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    zero_consumption="month_hour_average", 
    check=True,
    verbose=True,
)

xr.testing.assert_allclose(consumption_mix_direct, consumption_mix_xr)

consumption_mix_direct.shape

2026-05-18 16:01:21 Loading TYNDP pickle files
2026-05-18 16:01:22 Parsing hourly datetime index
2026-05-18 16:01:22 Loading technology and country mappings
2026-05-18 16:01:22 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\tyndp.py:1107: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-18 16:01:23 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\tyndp.py:297: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\tyndp.py:1169: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-18 16:01:23 Combining production and trade into Z_gross
2026-05-18 16:01:23 Built Z_gross with shape (8736, 512)
2026-05-18 16:01:23 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\tyndp.py:1273: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\tyndp.py:1293: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-18 16:01:23 Calculating direct production and trade shares
2026-05-18 16:01:23 Using 35 countries and 31 technologies
2026-05-18 16:01:23 Reshaping production shares
2026-05-18 16:01:25 Reshaping trade shares
2026-05-18 16:01:25 Checking direct shares sum to one
2026-05-18 16:01:25 Solving hourly trade system
2026-05-18 16:01:54 Imputing zero-consumption country-hours with month-hour averages
2026-05-18 16:01:54 Checking final consumption mix sums to one
2026-05-18 16:01:55 Building xarray DataArray
2026-05-18 16:02:12 Built consumption_mix_xr with shape (8736, 35, 35, 31)


(8736, 35, 35, 31)

## `premise` geography map

Map the FIONA source countries to the IAM regions used by premise. In the installed premise version, `remind` and `image` are available by default. `remind-eu` is included here to make availability explicit; if the topology is not installed, the column is filled with missing values and a warning is emitted.

In [14]:
source_countries = consumption_mix_xr.source_country.to_index()

premise_region_map = build_premise_region_map(
    source_countries,
    iam_models=("remind", "image", "remind-eu"),
    on_missing_model="warn",
)

premise_region_map.head()

Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'


,remind,image,remind-eu
country,,,
AL,NEU,CEU,NES
AT,EUR,WEU,EWN
BA,NEU,CEU,NES
BE,EUR,WEU,EWN
BG,EUR,CEU,ECS


In [15]:
premise_region_map.value_counts(dropna=False)

remind  image  remind-eu
EUR     CEU    ECE          6
               ECS          5
NEU     CEU    NES          5
EUR     WEU    EWN          4
               ENC          3
               ESC          3
               ESW          2
               UKI          2
NEU     WEU    NEN          2
EUR     CEU    ESC          1
        WEU    DEU          1
               FRA          1
Name: count, dtype: int64

### Quick REMIND-EU Check

If `remind-eu-topology.json` is available in `data/tyndp`, the `remind-eu` column should resolve to more detailed European regions instead of `<NA>`.

In [19]:
quick_remind_eu_map = build_premise_region_map(
    ["FR", "DE", "ES", "PT", "IT", "NL", "PL", "UK", "NO"],
    iam_models=("remind", "tiam-ucl", "message", "gcam", "image", "remind-eu",),
    on_missing_model="raise",
)

quick_remind_eu_map

Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'


,remind,tiam-ucl,message,gcam,image,remind-eu
country,,,,,,
DE,EUR,WEU,WEU,EU-15,WEU,DEU
ES,EUR,WEU,WEU,EU-15,WEU,ESW
FR,EUR,WEU,WEU,EU-15,WEU,FRA
IT,EUR,WEU,WEU,EU-15,WEU,ESC
NL,EUR,WEU,WEU,EU-15,WEU,EWN
NO,NEU,WEU,WEU,European Free Trade Association,WEU,NEN
PL,EUR,EEU,EEU,EU-12,CEU,ECE
PT,EUR,WEU,WEU,EU-15,WEU,ESW
UK,EUR,UK,WEU,EU-15,WEU,UKI


## Map technologies to `premise` activities

Use the TYNDP/electricity concordance matrix to aggregate ENTSO-E technologies into premise activities, then aggregate source countries into premise/IAM regions. Both mappings use xarray matrix multiplication to avoid materializing a large sparse broadcasted array.

In [22]:
premise_consumption_mix_mapper = PremiseConsumptionMixMapper(
    technology_mapping=technology_mapping,
    iam_model="remind-eu",
    on_missing_model="warn",
)
premise_activity_mix_xr = premise_consumption_mix_mapper.map_technologies(
    consumption_mix_xr,
    check=True,
)

premise_activity_mix_xr

<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     source_country: 35, premise_activity: 38)> Size: 3GB
array([[[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [4.36123539e-03, 0.00000000e+00, 1.00777551e-02, ...,
          0.00000000e+00, 0.00000000e+00, 4.36123539e-03],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.11777094e-02, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.11777094e-02],
         [3.89980702e-05, 0.00000000e+00, 1.16193729e-04, ...,
          0.00000000e+00, 0.00000000e+00, 3.89980702e-05]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.83902673e-02, 0.00000000e+00, 5.47933199e-02, ...,
          0.00000000e+00, 0.00000000e+00, 1.83902673e-02]]]])
Coordinates:
  * source_country    (source_country) object 280B 'AL' 'AT' 'BA' ... 'SK' 'UK'
  * time              (time) datetime64[ns] 70kB 2040-01-01 ... 2040-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * premise_activity  (premise_activity) object 304B 'electricity production,...

In [23]:
premise_region_mix_xr = premise_consumption_mix_mapper.map_regions(
    premise_activity_mix_xr,
    premise_region_map=premise_region_map,
    check=True,
)

premise_region_mix_xr

<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     premise_activity: 38, premise_region: 11)> Size: 1GB
array([[[[1.03015903e-04, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.03015903e-04, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[4.25269668e-05, 4.36123539e-03, 9.15873095e-04, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 1.00777551e-02, 5.35108226e-03, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 3.56400881e-05, 0.00000000e+00, ...,
          5.03027819e-07, 8.13787411e-06, 3.90581816e-05]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.84014963e-02],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.47933199e-02],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.84014963e-02]]]])
Coordinates:
  * time              (time) datetime64[ns] 70kB 2040-01-01 ... 2040-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * premise_activity  (premise_activity) object 304B 'electricity production,...
  * premise_region    (premise_region) object 88B 'NES' 'EWN' ... 'FRA' 'UKI'
Attributes:
    iam_model:  remind-eu

In [24]:
premise_region_mix = (
    premise_region_mix_xr
    .to_dataframe(name="share")
    .reset_index()
)

In [25]:
premise_region_mix.loc[premise_region_mix["consumer_country"].eq("ES")].groupby(
    ["premise_region", "premise_activity"],
    dropna=False,
)["share"].sum().nlargest(30) / 8736

premise_region  premise_activity                                                                                    
ESW             electricity production, wind, <1MW turbine, onshore                                                     0.222761
                electricity production, wind, >3MW turbine, onshore                                                     0.222761
                electricity production, photovoltaic, commercial                                                        0.128090
                electricity production, photovoltaic, residential                                                       0.128090
                electricity production, hydro, reservoir, alpine region                                                 0.054190
                electricity production, hydro, reservoir, non-alpine region                                             0.054190
                electricity production, wind, 1-3MW turbine, offshore                                        